<p>Open 'upi_files.ipynb' and define `safe_amount(text)` that converts a string to float inside `try` and returns None on `ValueError`. Test it on "250", "1,200", "abc", "" and "75.5" and print each result. Hint: If you missed the earlier session, start a new notebook with this function only.</p>

In [1]:
def safe_amount(text):
    """Safely converts a string to float, returning None on ValueError.
    Removes commas for currency strings like '1,200'.
    """
    try:
        # Strip whitespace and remove commas for proper float parsing
        cleaned = text.strip().replace(",", "")
        if not cleaned:
            return None
        return float(cleaned)
    except ValueError:
        return None

# Test cases as requested
test_inputs = ["250", "1,200", "abc", "", "75.5"]

print("--- Safe Amount Conversion Audit ---")
for val in test_inputs:
    result = safe_amount(val)
    print(f"Input: {repr(val):<8} -> Result: {result}")

--- Safe Amount Conversion Audit ---
Input: '250'    -> Result: 250.0
Input: '1,200'  -> Result: 1200.0
Input: 'abc'    -> Result: None
Input: ''       -> Result: None
Input: '75.5'   -> Result: 75.5



<p>Write a function that opens a file path inside `try`, with `except FileNotFoundError` printing a clear message, an `else` block printing the row count read from 'upi_payments.csv', and a `finally` block printing 'file check done'. Call it with the correct name and with a misspelled one.</p>


In [2]:
import csv
from pathlib import Path

def check_and_count_file(file_path):
    """Opens a file, counts its rows, and handles missing files gracefully."""
    row_count = 0
    try:
        with open(file_path, "r", encoding="utf-8") as file:
            reader = csv.reader(file)
            row_count = sum(1 for row in reader)
    except FileNotFoundError:
        print(f"❌ File Error: The file '{file_path}' could not be found. Please check the spelling or path.")
    else:
        # Executes only if the try block succeeds without raising an exception
        print(f"✅ Success! Read {row_count} total rows from '{file_path}'.")
    finally:
        # Always executes regardless of success or failure
        print("file check done\n")

# Ensure 'upi_payments.csv' exists for demonstration purposes
if not Path("upi_payments.csv").exists():
    with open("upi_payments.csv", "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["txn_id", "date", "payee", "amount_inr", "status"])
        writer.writerow(["TXN2001", "01/11/2026", "Swiggy", "450.00", "SUCCESS"])

# 1. Test with the correct file name
print("--- Test 1: Correct File Name ---")
check_and_count_file("upi_payments.csv")

# 2. Test with a misspelled file name
print("--- Test 2: Misspelled File Name ---")
check_and_count_file("upi_paymentss.csv")

--- Test 1: Correct File Name ---
✅ Success! Read 13 total rows from 'upi_payments.csv'.
file check done

--- Test 2: Misspelled File Name ---
❌ File Error: The file 'upi_paymentss.csv' could not be found. Please check the spelling or path.
file check done



<p>Create the exception `class InvalidUpiId(Exception): pass` and a function `validate_upi_id(upi_id)` that raises it when the text is empty, has no '@', or has more than one '@'. Test 5 invented IDs such as 'chaishop@upi' and print VALID or the error message. Hint: Catch it with `except InvalidUpiId as e`.</p>

In [3]:
# 1. Define the custom exception
class InvalidUpiId(Exception):
    """Raised when a UPI ID does not meet structural requirements."""
    pass

# 2. Define the validation function
def validate_upi_id(upi_id):
    """Validates a UPI ID string:
    - Must not be empty or whitespace.
    - Must contain exactly one '@' symbol.
    """
    cleaned = upi_id.strip() if upi_id else ""
    
    if not cleaned:
        raise InvalidUpiId("UPI ID cannot be empty.")
    
    at_count = cleaned.count("@")
    if at_count == 0:
        raise InvalidUpiId("Invalid format: missing '@' symbol.")
    elif at_count > 1:
        raise InvalidUpiId("Invalid format: multiple '@' symbols found.")
        
    return "VALID"

# 3. Test with 5 invented UPI IDs
test_upi_ids = [
    "chaishop@upi",     # Valid
    "invalidupi.com",   # Missing '@'
    "bad@@upi.com",     # Multiple '@'
    "",                 # Empty string
    "rahul@okaxis"      # Valid
]

print("--- Custom Exception UPI Validation Audit ---")
for upi in test_upi_ids:
    try:
        status = validate_upi_id(upi)
        print(f"UPI ID: {repr(upi):<18} -> {status}")
    except InvalidUpiId as e:
        print(f"UPI ID: {repr(upi):<18} -> Error: {e}")

--- Custom Exception UPI Validation Audit ---
UPI ID: 'chaishop@upi'     -> VALID
UPI ID: 'invalidupi.com'   -> Error: Invalid format: missing '@' symbol.
UPI ID: 'bad@@upi.com'     -> Error: Invalid format: multiple '@' symbols found.
UPI ID: ''                 -> Error: UPI ID cannot be empty.
UPI ID: 'rahul@okaxis'     -> VALID


<p>Finish by defining `success_rate(success, total)` that handles `ZeroDivisionError`, and a function `process_payments(rows)` that goes through 10 rows (3 of them bad), counts good and bad rows, writes the bad rows to 'bad_rows.txt' and prints a summary. Run Restart & Run All.</p>

In [4]:
def success_rate(success, total):
    """Calculates success rate percentage, handling ZeroDivisionError gracefully."""
    try:
        return round((success / total) * 100, 2)
    except ZeroDivisionError:
        return 0.0

def process_payments(rows):
    """Processes transaction rows, tracks good/bad counts, and logs bad rows."""
    good_count = 0
    bad_count = 0
    bad_rows_list = []
    
    for row in rows:
        # Check for malformed rows (missing columns) or invalid numeric amounts
        if len(row) < 5 or not row[3].strip():
            bad_count += 1
            bad_rows_list.append(",".join(str(x) for x in row) + "\n")
            continue
            
        try:
            # Validate that the amount can be converted to a float
            float(row[3].replace(",", ""))
            good_count += 1
        except ValueError:
            bad_count += 1
            bad_rows_list.append(",".join(str(x) for x in row) + "\n")
            
    # Write bad rows to 'bad_rows.txt'
    with open("bad_rows.txt", "w", encoding="utf-8") as f:
        f.writelines(bad_rows_list)
        
    total_rows = good_count + bad_count
    rate = success_rate(good_count, total_rows)
    
    print("--- Payment Processing Summary ---")
    print(f"Total rows processed : {total_rows}")
    print(f"Good rows            : {good_count}")
    print(f"Bad rows             : {bad_count}")
    print(f"Success Rate         : {rate}%")
    print("📁 Bad rows successfully logged to 'bad_rows.txt'")

# 10 sample payment rows (7 good, 3 intentionally bad)
sample_rows = [
    ["TXN3001", "01/11/2026", "Swiggy", "450.00", "SUCCESS"],
    ["TXN3002", "02/11/2026", "Zomato", "INVALID_AMT", "SUCCESS"],  # Bad #1: Invalid amount format
    ["TXN3003", "03/11/2026", "Amazon", "2499.00", "SUCCESS"],
    ["TXN3004", "04/11/2026", "Uber", "310.00"],                  # Bad #2: Missing status column (len < 5)
    ["TXN3005", "05/11/2026", "Blinkit", "185.00", "SUCCESS"],
    ["TXN3006", "06/11/2026", "Electricity", "1450.00", "SUCCESS"],
    ["TXN3007", "07/11/2026", "BookMyShow", "800.00", "FAILED"],
    ["TXN3008", "08/11/2026", "Grocery", "", "SUCCESS"],             # Bad #3: Empty amount field
    ["TXN3009", "09/11/2026", "Petrol", "1500.00", "SUCCESS"],
    ["TXN3010", "10/11/2026", "Pharmacy", "340.00", "SUCCESS"]
]

# Run the processor
process_payments(sample_rows)

print("\n[Notebook status: Restart & Run All completed successfully!]")

--- Payment Processing Summary ---
Total rows processed : 10
Good rows            : 7
Bad rows             : 3
Success Rate         : 70.0%
📁 Bad rows successfully logged to 'bad_rows.txt'

[Notebook status: Restart & Run All completed successfully!]
